# الدرس العاشر: التدخل البشري والتحكم في العمليات الحساسة (Human-in-the-Loop Middleware)

## المقدمة والاهداف التعليمية
في هذا الدفتر، سنتعلم كيفية تطبيق بروتوكول التدخل والموافقة البشرية (Human-in-the-Loop - HITL) في وكلاء LangChain 1.x لحماية الانظمة من القرارات غير المصرح بها.

## لماذا نحتاج الى التدخل البشري (HITL)؟
عندما يتمتع الوكيل بصلاحيات تنفيذية حقيقية مثل:
- تحويل الاموال او الخصم من البطاقات الائتمانية.
- تعديل او حذف سجلات في قواعد البيانات الانتاجية (`DROP TABLE`, `DELETE`).
- ارسال رسائل بريد الكتروني رسمية للعملاء.

يجب الا يُترك اتخاذ القرار النهائي للنموذج بنسبة 100%. توفر `HumanInTheLoopMiddleware` نقطة توقف واعتراض (Breakpoint / Interruption) قبل تنفيذ الاداة الحساسة، بانتظار قرار المشرف البشري بالموافقة او التعديل او الرفض.

## الخطوة 1: استيراد الاعتماديات وتهيئة البيئة
نقوم باستيراد `HumanInTheLoopMiddleware` ونموذج الدردشة الموحد.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")

model = init_chat_model("openai/gpt-oss-120b", model_provider="groq", temperature=0)

## الخطوة 2: تعريف ادوات عادية وادوات حساسة
نعرف اداتين: الاولى استعلامية غير ضارة، والثانية تنفيذية حساسة تتطلب موافقة صريحة قبل التشغيل.

In [ ]:
@tool
def get_account_balance(account_id: str) -> str:
    """Query current account balance."""
    return f"Account {account_id} current balance is $25,000.00."

@tool
def transfer_funds(account_id: str, recipient_id: str, amount: float) -> str:
    """Execute financial money transfer between accounts. CRITICAL OPERATION."""
    return f"Successfully transferred ${amount:,.2f} from {account_id} to {recipient_id}."

print("Tools registered:")
print("1. Safe tool:", get_account_balance.name)
print("2. Sensitive tool:", transfer_funds.name)

## الخطوة 3: تكوين برمجية التدخل البشري `HumanInTheLoopMiddleware`
نقوم بتعريف قاموس `interrupt_on` لتحديد الادوات التي تتطلب تعليق التنفيذ والموافقة البشرية.

In [ ]:
# تحديد الادوات الحساسة التي توجب ايقاف التنفيذ
hitl_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "transfer_funds": True  # طلب الموافقة البشرية قبل تنفيذ التحويل المالي
    },
    description_prefix="Supervisor Authorization Required"
)

print("HumanInTheLoopMiddleware configured for: transfer_funds")

## الخطوة 4: انشاء الوكيل وفحص العمليات الآمنة
ننشئ الوكيل ونفحص استدعاء الاداة الآمنة `get_account_balance` حيث تعمل دون اي اعتراض.

In [ ]:
agent = create_agent(
    model=model,
    tools=[get_account_balance, transfer_funds],
    system_prompt="You are a secure banking assistant. Execute operations requested by authenticated users.",
    middleware=[hitl_middleware]
)

# اختبار العملية الآمنة
safe_response = agent.invoke({
    "messages": [
        ("user", "What is the balance of account ACC-1002?")
    ]
})

print("Safe Tool Execution Result:")
print(safe_response["messages"][-1].content)

## الخطوة 5: آلية عمل الاعتراض البشري اثناء محاولة تنفيذ الاداة الحساسة
عندما يطلب المستخدم تحويل الاموال، تعترض البرمجية الوسيطة العملية وتولد اشعارا يوضح تفاصيل المعاملة والمدخلات بانتظار تأكيد المسؤول.

In [ ]:
# محاكاة سيناريو طلب عملية حساسة
print("Architecture Flow for Critical Operations:")
print("1. User submits request: 'Transfer $5,000 from ACC-1002 to ACC-9900'")
print("2. Model recognizes intent and generates tool_call: transfer_funds(amount=5000)")
print("3. HumanInTheLoopMiddleware intercepts the call BEFORE execution.")
print("4. Graph state transitions to INTERRUPTED state.")
print("5. Supervisor reviews parameters (Amount, Source, Target).")
print("6. If Approved -> Graph resumes and executes transfer.")
print("7. If Rejected -> Tool call canceled and explanatory feedback returned to user.")